# 04 · Reshape and transpose real images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/04-reshape-and-transpose.ipynb)

*Part III · exercise · 15 min*

> 🇪🇸 **Reshape y transposición de imágenes reales** — De HWC a CHW, de NHWC a NCHW, y por qué reshape destruye una imagen en silencio.

HWC to CHW, NHWC to NCHW, and why reshape silently destroys an image.

## What you will be able to do

- Convert an image between `(H, W, C)` and `(C, H, W)` with `np.transpose`.
- Convert a batch between NHWC and NCHW, and know which axis is which when two share a size.
- Explain why `reshape` runs without error and still destroys the image.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
from skimage import data

photo = data.immunohistochemistry()   # (512, 512, 3) real histology
cells = data.cell()                   # (660, 550)    real microscopy, grayscale
print(photo.shape, cells.shape)

## Why this matters

> 🇪🇸 Los microscopios y las cámaras ordenan sus ejes según el hardware, no
> según lo que el modelo espera. Equivocarse no da error: el modelo funciona con
> datos revueltos y devuelve resultados seguros y sin sentido.

Microscopes and cameras order their axes according to the hardware, not
according to what a model expects. Getting this wrong does not crash — the model
runs on scrambled data and returns confident, meaningless output.

In a drug screen, that is a wrong decision about whether a compound works. The
famous version in tech: a model trained in TensorFlow (`NHWC`) deployed into
PyTorch (`NCHW`) with no transpose.

## Exercise 1 — one image, two orderings

> 🇪🇸 Una imagen, dos ordenaciones de ejes.

In [ ]:
# TODO 1: Print both shapes. Which one has no colour axis?

# TODO 2: Convert `photo` from (H, W, C) to (C, H, W) with np.transpose.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
print(photo.shape, cells.shape)   # (512, 512, 3) (660, 550)
# `cells` is grayscale — order 2, no colour axis at all.

chw = np.transpose(photo, (2, 0, 1))      # (3, 512, 512) — correct
print(chw.shape)

# The tuple (2, 0, 1) reads: "the new axis 0 is the old axis 2, the new axis 1
# is the old axis 0, the new axis 2 is the old axis 1."

## Exercise 2 — a batch, and two axes of the same size

> 🇪🇸 Un lote, y dos ejes del mismo tamaño.

In [ ]:
# TODO 3: Stack `photo` three times into a batch of shape (3, 512, 512, 3).
#         Which axis is the batch axis?

# TODO 4: Convert that batch from NHWC to NCHW -> (3, 3, 512, 512).
#         Two axes now both have size 3. How do you know which is which?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
batch = np.stack([photo, photo, photo])      # (3, 512, 512, 3)
print(batch.shape)                            # axis 0 is the batch axis

nchw = np.transpose(batch, (0, 3, 1, 2))     # (3, 3, 512, 512)
print(nchw.shape)

# You know which is which ONLY because you wrote the transpose. Nothing in the
# array records it. Check it by hand — a batch axis and a colour axis behave
# differently under indexing:
print(np.array_equal(nchw[0], nchw[1]))      # True  — the 3 stacked copies
print(np.array_equal(nchw[:, 0], nchw[:, 1]))  # False — the 3 colour channels

## Exercise 3 — the one that runs and is still wrong

> 🇪🇸 El que se ejecuta sin error y aun así está mal.

In [ ]:
# TODO 5: photo.reshape(3, 512, 512) runs WITHOUT error but is wrong.
#         Run it, compare against TODO 2, and explain the difference.
#         If you can, display both with matplotlib and look at them.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
chw   = np.transpose(photo, (2, 0, 1))      # (3, 512, 512) — correct
wrong = photo.reshape(3, 512, 512)          # (3, 512, 512) — runs, but scrambles

print(chw.shape == wrong.shape)             # True  — identical shapes
print(np.array_equal(chw, wrong))           # False — completely different data

# Optional, and worth doing once:
# import matplotlib.pyplot as plt
# fig, ax = plt.subplots(1, 2, figsize=(8, 4))
# ax[0].imshow(chw[0],   cmap="gray"); ax[0].set_title("transpose — a channel")
# ax[1].imshow(wrong[0], cmap="gray"); ax[1].set_title("reshape — nonsense")

## What just happened

**Reshape only reinterprets numbers in memory order. Transpose moves them
according to axis meaning.** Both give shape `(3, 512, 512)`; only one is the
image.

> 🇪🇸 `reshape` reinterpreta los números en el orden en que están en memoria;
> `transpose` los mueve según el significado de cada eje.

And TODO 4 makes the deeper point: **once two axes share a size, the shape
cannot tell you which is which.** Only your own tracking can. No error will be
raised, no shape will look wrong, and the model will train — on scrambled data.

That is everything the first Kahoot asks about.

---

## Time for Kahoot 🎯

**Kahoot 1 — Tensor Vocabulary & Shapes** · 6 questions, about 5 minutes.

> 🇪🇸 **Vocabulario de tensores y formas** — 6 preguntas, unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-1)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_1_vocabulary_shapes.xlsx)

Next up: **05 · Video pipeline design** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/05-video-pipeline-design.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)